# Wolf & Badger — ESG data exploration (SAM 503 · Lab 4)
Explore, clean and visualise the data behind the Streamlit dashboard. Runs in **Google Colab** with no setup.

**Steps:** load → inspect → clean/reshape → calculate → visualise → scenario check.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Hello from Colab!")

## 1 · Load the data
Set `GITHUB_USER` to your username after pushing the repo. If the download fails, the notebook falls back to a local `data/` folder.

In [ ]:
GITHUB_USER = "linshiaoting-glitch"
REPO = "wolf-badger-esg-dashboard"
BASE = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main/data/"

def read(name):
    try:
        return pd.read_csv(BASE + name)
    except Exception:
        return pd.read_csv("data/" + name)   # local fallback (or upload the CSVs to Colab)

em  = read("emissions_annual.csv")
s3  = read("scope3_categories.csv")
bia = read("b_impact_scores.csv")
sg  = read("social_governance.csv")
kq  = read("kpi_quality_lab1.csv")
em

## 2 · Inspect

In [ ]:
for name, df in {"emissions": em, "scope3": s3, "b_impact": bia, "social_gov": sg, "kpi_quality": kq}.items():
    print(f"{name:12s} rows={len(df):3d}  missing values={int(df.isna().sum().sum())}")
em.dtypes

## 3 · Clean & reshape
- Keep the **restated** FY2023 and the reported FY2024 as the comparable series.
- Check that the scopes add up to the reported total.

In [ ]:
cmp = em[em.version.isin(["Restated", "Reported"])].set_index("year")
cmp["sum_s1_s2mkt_s3"] = cmp[["scope1_t", "scope2_market_t", "scope3_t"]].sum(axis=1)
cmp["sum_s1_s2loc_s3"] = cmp[["scope1_t", "scope2_location_t", "scope3_t"]].sum(axis=1)
cmp[["total_reported_t", "sum_s1_s2mkt_s3", "sum_s1_s2loc_s3"]]

> **Finding:** for FY2024 the reported total (10,330) equals S1 + **location-based** S2 + S3, not the market-based sum (10,300).

## 4 · Calculate growth and intensity

In [ ]:
yoy = cmp[["total_reported_t", "scope3_t", "gmv_gbp_m"]].pct_change().iloc[-1]
print("YoY change FY2023→FY2024:"); print((yoy * 100).round(1).astype(str) + "%")
cmp["intensity_check"] = cmp.total_reported_t / cmp.gmv_gbp_m
cmp[["intensity_t_per_gbp_m", "intensity_check"]].round(1)

## 5 · Visualise

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
lbl = ["FY2023 restated", "FY2024"]
ax[0].bar(lbl, cmp.scope2_market_t, label="Scope 2 (MB)", color="#4F7FA8")
ax[0].bar(lbl, cmp.scope3_t, bottom=cmp.scope2_market_t, label="Scope 3", color="#2F6B4F")
ax[0].set_title("Emissions by scope (tCO₂e)"); ax[0].legend()

piv = s3.pivot(index="year", columns="category", values="tco2e")
piv.plot(kind="bar", stacked=True, ax=ax[1], color=["#8DB59E", "#2F6B4F", "#E2C792"], rot=0)
ax[1].set_title("Scope 3 by category (tCO₂e)"); ax[1].legend(fontsize=7)

areas = ["Governance", "Workers", "Community", "Environment", "Customers"]
b = bia[bia.impact_area.isin(areas)].pivot(index="impact_area", columns="assessment_year", values="score").loc[areas]
b.plot(kind="bar", ax=ax[2], color=["#E2C792", "#2F6B4F"], rot=20)
ax[2].set_title("B Impact score by area")
plt.tight_layout(); plt.show()

## 6 · Data quality (from Lab 1)

In [ ]:
summary = kq.groupby("pillar").agg(kpis=("kpi", "count"),
                                   disclosed=("quality_score", lambda s: (s > 0).sum()),
                                   avg_score=("quality_score", "mean")).round(1)
summary["coverage"] = (summary.disclosed / summary.kpis * 100).round(0).astype(int).astype(str) + "%"
summary

## 7 · Scenario check — what does a 2030 target imply?
Required annual fall in intensity, *r*, so that emissions reach the target while GMV grows at *g*:
$E_{2030} = E_{2024}\,(1+g)^6(1-r)^6$

In [ ]:
E24 = cmp.loc[2024, "total_reported_t"]
target = cmp.loc[2023, "total_reported_t"] * (1 - 0.42)       # illustrative −42% vs 2023
for g in (0.05, 0.10, 0.15):
    r = 1 - (target / E24) ** (1 / 6) / (1 + g)
    print(f"GMV growth {g:.0%}/yr -> intensity must fall {r:.1%}/yr to reach {target:,.0f} t by 2030")

## 8 · Next step
Run the dashboard: `streamlit run app.py` (locally), or deploy from GitHub on share.streamlit.io.